In [0]:
%run ./04_Function_Book

In [0]:
# Define dq_config: specifies all quality rules for payments (null, format, datatype, enum checks, PK)
# Used as input to the metrics generator

dq_config = {
    "table_name": "payments",

    "null_checks": {
        "order_id": 0,
        "payment_sequential": 0,
        "payment_type": 0,
        "payment_installments": 0,
        "payment_value": 0,
    },

    # FORMAT CHECKS
    "format_checks": {

        # REGEX CHECKS
        "regex": {
            "order_id": {
                "pattern": r"^[A-Za-z0-9]{12,}$",
                "threshold": 0
            },
            "payment_type": {
                "pattern": r"^[A-Za-z_]+$",
                "threshold": 0
            }
        },

        # DATATYPE CHECKS
        "datatype_check": {
            "order_id": {
                "expected": "string",
                "threshold": 0
            },
            "payment_sequential": {
                "expected": "int",
                "threshold": 0
            },
            "payment_type": {
                "expected": "string",
                "threshold": 0
            },
            "payment_installments": {
                "expected": "int",
                "threshold": 0
            },
            "payment_value": {
                "expected": "double",
                "threshold": 0
            },
        },
    },

    # ENUM CHECKS    
    "enum_checks": {
        "payment_type": {
            "allowed_values": ['boleto', 'credit_card', 'debit_card', 'not_defined', 'voucher'],
            "threshold": 0
        }
    },    

    # PRIMARY KEY CHECK
    "primary_key": {
        "column": ["order_id", "payment_sequential"],
        "threshold": 0
    }
}


In [0]:
# Define a function to apply quality checks and aggregate metrics for payments
# Applies null, format, enum, and PK checks; tags results with table name and timestamp
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

def generate_metrics_df(df, config):
    all_results = []
    all_results += run_null_checks(df, config)
    all_results += run_format_checks(df, config)
    all_results += run_enum_checks(df, config)
    all_results += run_pk_check(df, config)
    for result in all_results:
        result["table_name"] = config["table_name"]
    metrics_df = spark.createDataFrame(Row(**r) for r in all_results) \
        .withColumn("run_time", current_timestamp())
    return metrics_df


In [0]:
# Load the payments silver table and apply all configured data quality checks
# Produces metrics DataFrame for downstream aggregation and monitoring
df_payments = spark.table("retail_catalog.silver.payments")
metrics_df = generate_metrics_df(df_payments, dq_config)

In [0]:
# Return metrics DataFrame for use in master/orchestration notebook
metrics_df

DataFrame[column_name: string, check_name: string, metric_type: string, metric_value: bigint, threshold: bigint, status: string, table_name: string, run_time: timestamp]